In [1]:
import os
import re
import time
from glob import iglob
from io import BytesIO

import streamlit as st
import torch
from einops import rearrange
from fire import Fire
from PIL import ExifTags, Image
from st_keyup import st_keyup
from torchvision import transforms
from transformers import pipeline

from flux.cli import SamplingOptions
from flux.sampling import denoise, get_noise, get_schedule, prepare, unpack
from flux.util import (
    configs,
    embed_watermark,
    load_ae,
    load_clip,
    load_flow_model,
    load_t5,
)

/home/liudq/miniconda3/envs/flux/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_device = torch.device(device)
offload = False
output_dir = "output"


In [7]:
name = 'flux-dev'
is_schnell = name == "flux-schnell"

t5 = load_t5(device, max_length=256 if is_schnell else 512)
clip = load_clip(device)
model = load_flow_model(name, device="cpu" if offload else device)
ae = load_ae(name, device="cpu" if offload else device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


KeyboardInterrupt: 

In [ ]:

name = st.selectbox("Which model to load?", names)

is_schnell = name == "flux-schnell"
model, ae, t5, clip, nsfw_classifier = get_models(
    name,
    device=torch_device,
    offload=offload,
    is_schnell=is_schnell,
)

do_img2img = (
    st.checkbox(
        "Image to Image",
        False,
        disabled=is_schnell,
        help="Partially noise an image and denoise again to get variations.\n\nOnly works for flux-dev",
    )
    and not is_schnell
)
if do_img2img:
    init_image = get_image()
    if init_image is None:
        st.warning("Please add an image to do image to image")
    image2image_strength = st.number_input("Noising strength", min_value=0.0, max_value=1.0, value=0.8)
    if init_image is not None:
        h, w = init_image.shape[-2:]
        st.write(f"Got image of size {w}x{h} ({h*w/1e6:.2f}MP)")
    resize_img = st.checkbox("Resize image", False) or init_image is None
else:
    init_image = None
    resize_img = True
    image2image_strength = 0.0